# Datathon 2026 | Final: Hướng dẫn Kết nối và Nạp Dữ liệu

Notebook này hướng dẫn các đội thi cách truy cập vào bộ dữ liệu thực tế của Nhà Tốt (52 GB) từ Google Cloud Storage.

## Quy định bắt buộc:

1.  **Môi trường:** Thí sinh bắt buộc chỉ sử dụng **Google Colab** để thao tác với dữ liệu nhằm tối ưu tốc độ và tránh phát sinh chi phí truyền tải dữ liệu (Egress fee) cho BTC.
2.  **Xác thực:** Bạn phải sử dụng tài khoản **Gmail đã đăng ký với BTC** để được cấp quyền truy cập.

# Cấu trúc dữ liệu trên Google Cloud Storage

**Bucket Name:** `gs://datathon_2026_final/`

Dữ liệu được tổ chức theo cấu trúc cây thư mục (Object Prefixes) như sau:

```text
datathon_2026_final/
├── test/
│   └── test_users.parquet              # Danh sách 161,568 user_id cần dự đoán
└── train/
    ├── dim_listing/                    # Thông tin danh mục bất động sản (40 files)
    │   ├── part-00000-....parquet
    │   └── ...
    ├── fact_listing_snapshot/          # Hiệu suất tin đăng theo ngày (62 files)
    │   ├── part-00000-....parquet
    │   └── ...
    ├── fact_post_contact_interactions/ # Tương tác sau khi liên hệ (147 files)
    │   ├── part-00000-....parquet
    │   └── ...
    └── fact_user_events/               # Nhật ký hành vi người dùng (500 files)
        ├── datathon_fact_user_events-000000000000.parquet
        ├── datathon_fact_user_events-000000000104.parquet
        └── ...

In [ ]:
# 1. Xác thực tài khoản Google (Dùng email đã đăng ký với BTC)
from google.colab import auth
auth.authenticate_user()

# 2. Cấu hình Project ID và đường dẫn lưu trữ
BUCKET_NAME = "datathon_2026_final"

TRAIN_PATH = f"gs://{BUCKET_NAME}/train/"
TEST_PATH = f"gs://{BUCKET_NAME}/test/"

In [ ]:
# 2. Đọc thử file test để kiểm tra kết nối
import pandas as pd
# GCP tự động nhận diện thông qua credentials vừa đăng nhập
df_test = pd.read_parquet(f"{TEST_PATH}test_users.parquet")

print("Kết nối thành công! Số lượng user cần dự đoán:", len(df_test))
print(df_test.head())

Kết nối thành công! Số lượng user cần dự đoán: 161568
                                             user_id
0  c9911d7e40a8c8b7ffd257e21177fcd18b56ce6fa10612...
1  612e00627e215ac3086d919c6a9dc5aa1db5850bbe59d2...
2  f1b4015a5dc2884da1152fc097642b2cb3b91979d3cf61...
3  bab7641e6e2a625a6a3ae924f168a72324339943b1b4ec...
4  0f76d272d5d06e84d34cbee06d9228af42f6a1ff242f46...


In [ ]:
import pandas as pd

# =====================================================================
# Load thử 1 file Parquet đơn lẻ để xem nhanh dữ liệu (An toàn cho RAM)
# =====================================================================
print("--- TEST ĐỌC 1 FILE ---")
single_file_path = f"{TRAIN_PATH}fact_user_events/datathon_fact_user_events-000000000104.parquet"

df_sample = pd.read_parquet(single_file_path)
print(f"Load thành công! Số dòng trong file này: {len(df_sample)}")
display(df_sample.head())

--- TEST ĐỌC 1 FILE ---
Load thành công! Số dòng trong file này: 324143


,is_login,user_id,session_id,event_id,item_id,city_name,category,event_type,query,event_ts,surface,position,device,dwell_time_sec,is_contact,date
0,login,d128af56deaf365da0e4b9e9dc2ed4c91b9dec9217dd9d...,f5ac97a0f36ee3cc3b56fd1c263e2cd4632cd57a0be0b8...,648024e2e8774099e4da4b402eb2f49e8a76e74b67c11f...,178dde9314ad8a90112c0a232e0c28ca33b91ff03f3e4b...,Tp Hồ Chí Minh,1020,other_interaction,None,2025-11-09 14:04:00.945047,ad_view,NaN,Android,NaN,1,2025-11-09
1,non-login,bf7f5d09de94c2cc9c718b8e3939bc666e2420a96d1aec...,20482e9835a6707b667f1122a92c7ccb36c5869f5af9f4...,5ff387de2c677f7e3c20d6e205df669851e8d9c969d74a...,c155a1c00892e4363267b35833b9454340ba77fdb5da9c...,Bình Dương,1020,other_interaction,None,2025-11-09 20:42:37.730002,ad_view,NaN,Android,NaN,1,2025-11-09
2,login,3701575dae5028ecb284d43cc7dc28f71bb7d93e868259...,6317a0a1b3a8ad2d87d5a10eaa9be5b47802a86b57b1e6...,3b60a9e5266ecb218e17890c6b8d0fa8abaddcb281d4c3...,c8ec5b6bcd015aaa61c333435c9f1cfb61cf415f25bb8f...,Tp Hồ Chí Minh,1030,other_interaction,None,2025-11-09 16:28:05.596003,ad_view,NaN,Android,NaN,1,2025-11-09
3,login,663dfe4ec6339d8639a567d99f7dfb1be95b598050c6f4...,5a34e578e0d2c3ba20e274e09760ea837f38db168a1545...,4aec2e6edf5f621ca6146bde63a4a52cb6cc648e331f6b...,4245fdad8221c3bdeb52174b4a27d3ff7eff09b62eed1b...,Lâm Đồng,1040,other_interaction,None,2025-11-09 15:26:02.010001,ad_view,NaN,Android,NaN,1,2025-11-09
4,login,23cf693c46ab4ba0a704028b1e0c9518c74e34d06dfb2a...,2ca625168e3427ec116f73d86031dc48e0bc4a4091d0a3...,37e9f8c3114e3a22339b438d6069805cecf6c180e9e64c...,32b99a5d852e0a6844f45b5b8ebf6db368a72868ed81ce...,Tp Hồ Chí Minh,1020,other_interaction,None,2025-11-09 16:09:01.970085,ad_view,NaN,Android,NaN,1,2025-11-09


Bảng nhật ký hành vi người dùng (fact_user_events) chứa hơn 161 triệu dòng sự kiện được chia thành 500 file Parquet. Để tránh lỗi tràn bộ nhớ (Out-of-Memory), thí sinh nên sử dụng pyarrow.dataset để quét thư mục và chỉ lọc lấy những dữ liệu cần thiết.


In [ ]:
import pyarrow.dataset as ds
# =====================================================================
# Quét toàn bộ thư mục bằng PyArrow Dataset (Khuyên dùng cho Big Data)
# =====================================================================
print("\n--- TEST KẾT NỐI TOÀN BỘ THƯ MỤC ---")
folder_path = f"{TRAIN_PATH}fact_user_events/"

# Lệnh này chỉ tạo connection (mapping) tới các file, KHÔNG load hết vào RAM
dataset = ds.dataset(folder_path, format="parquet")

print(f"Hệ thống đã nhận diện được {len(dataset.files)} files trong thư mục.")
print("Lấy thử 5 dòng đầu tiên từ toàn bộ dataset:")
display(dataset.head(5).to_pandas())


--- TEST KẾT NỐI TOÀN BỘ THƯ MỤC ---
Hệ thống đã nhận diện được 500 files trong thư mục.
Lấy thử 5 dòng đầu tiên từ toàn bộ dataset:


,is_login,user_id,session_id,event_id,item_id,city_name,category,event_type,query,event_ts,surface,position,device,dwell_time_sec,is_contact,date
0,login,a02289fe8e1e16c9b5f801667d2827e68f294032d0b198...,e8ad47cde55561f7bda9266683bbc42cc3834c0156eaf1...,4ad71326e8a0f735af644dabe6a660110d63fcff7392b7...,2f68e2ccabed3bdfbf9a6f5ce5e125ebcbabbdb45cae59...,Tp Hồ Chí Minh,1020,other_interaction,None,2025-11-09 15:53:05.305008,ad_view,NaN,Android,NaN,1,2025-11-09
1,login,c78e29d88cd3bcff4b531769195635f3395a4b0990eb92...,e2e725fb1c108a18d8cc55625684c74d69be0c473c3a2b...,1e48192d10b753368993858d076409b9f0a70955bf6529...,97988594504114e1f2da13f1600d6ce6fdb059a11c329d...,Tp Hồ Chí Minh,1040,other_interaction,None,2025-11-09 16:15:49.863004,ad_view,NaN,Android,NaN,1,2025-11-09
2,login,6d7db41749e69ad564948cab6284a7292e3eef1bbfa39e...,63b17940c17d6d7a83ac72894cd82a84240c0b8175a7fd...,73305b6e7a57a8222e93965e63aad3027b3a09b81a8f5c...,d432d5a76b304e29c72cf11eda87b73f78205daa09be1d...,Đồng Nai,1020,other_interaction,None,2025-11-09 13:19:32.768003,ad_view,NaN,Android,NaN,1,2025-11-09
3,login,c58edefd76dac6ab84486002de05dcc9264d92b080f992...,988b36a9cada4f2e441542ab49d7dbbcce4cfa8e442d5b...,84c5f9942e24c5d3e82190d672fba39ab5c5dcf3cf8278...,b74c89b1d6de85bff307820d879078d7dc619f1c6f8b7b...,Tp Hồ Chí Minh,1050,other_interaction,None,2025-11-09 18:23:52.006005,ad_view,NaN,Android,NaN,1,2025-11-09
4,login,285b088ff710612ae598f24ffad9b147f6f017c7b9342f...,92a603a03a1a78ba939c85e9bf8a5d4f37b38d7edd5dab...,eee1dcf790558591206abf14fff21fc95c97e7e4d29adb...,8be4baf58ed281f4396a3cfc7067eb7ffb501e85a0a31d...,Tp Hồ Chí Minh,1010,other_interaction,None,2025-11-09 13:59:56.191006,ad_view,NaN,Android,NaN,1,2025-11-09


In [ ]:
single_file = (
    f"{TRAIN_PATH}fact_user_events/datathon_fact_user_events-000000000104.parquet"
)

df_sample = pd.read_parquet(single_file)
print(f"So dong trong file: {len(df_sample)}")
df_sample.head()

So dong trong file: 324143


,is_login,user_id,session_id,event_id,item_id,city_name,category,event_type,query,event_ts,surface,position,device,dwell_time_sec,is_contact,date
0,login,d128af56deaf365da0e4b9e9dc2ed4c91b9dec9217dd9d...,f5ac97a0f36ee3cc3b56fd1c263e2cd4632cd57a0be0b8...,648024e2e8774099e4da4b402eb2f49e8a76e74b67c11f...,178dde9314ad8a90112c0a232e0c28ca33b91ff03f3e4b...,Tp Hồ Chí Minh,1020,other_interaction,None,2025-11-09 14:04:00.945047,ad_view,NaN,Android,NaN,1,2025-11-09
1,non-login,bf7f5d09de94c2cc9c718b8e3939bc666e2420a96d1aec...,20482e9835a6707b667f1122a92c7ccb36c5869f5af9f4...,5ff387de2c677f7e3c20d6e205df669851e8d9c969d74a...,c155a1c00892e4363267b35833b9454340ba77fdb5da9c...,Bình Dương,1020,other_interaction,None,2025-11-09 20:42:37.730002,ad_view,NaN,Android,NaN,1,2025-11-09
2,login,3701575dae5028ecb284d43cc7dc28f71bb7d93e868259...,6317a0a1b3a8ad2d87d5a10eaa9be5b47802a86b57b1e6...,3b60a9e5266ecb218e17890c6b8d0fa8abaddcb281d4c3...,c8ec5b6bcd015aaa61c333435c9f1cfb61cf415f25bb8f...,Tp Hồ Chí Minh,1030,other_interaction,None,2025-11-09 16:28:05.596003,ad_view,NaN,Android,NaN,1,2025-11-09
3,login,663dfe4ec6339d8639a567d99f7dfb1be95b598050c6f4...,5a34e578e0d2c3ba20e274e09760ea837f38db168a1545...,4aec2e6edf5f621ca6146bde63a4a52cb6cc648e331f6b...,4245fdad8221c3bdeb52174b4a27d3ff7eff09b62eed1b...,Lâm Đồng,1040,other_interaction,None,2025-11-09 15:26:02.010001,ad_view,NaN,Android,NaN,1,2025-11-09
4,login,23cf693c46ab4ba0a704028b1e0c9518c74e34d06dfb2a...,2ca625168e3427ec116f73d86031dc48e0bc4a4091d0a3...,37e9f8c3114e3a22339b438d6069805cecf6c180e9e64c...,32b99a5d852e0a6844f45b5b8ebf6db368a72868ed81ce...,Tp Hồ Chí Minh,1020,other_interaction,None,2025-11-09 16:09:01.970085,ad_view,NaN,Android,NaN,1,2025-11-09


In [ ]:
folder_path = f"{TRAIN_PATH}fact_user_events/"

# Lenh nay chi tao connection (mapping) toi cac file,
# KHONG load het vao RAM
dataset = ds.dataset(folder_path, format="parquet")

# Vi du: chi lay cac su kien tich cuc cua mot tap user nho
import pyarrow.compute as pc

POSITIVE = ["view_phone", "contact_chat", "contact_zalo", "contact_sms"]
filtered = dataset.to_table(
    columns=["user_id", "item_id", "event_type", "event_ts"],
    filter=pc.field("event_type").isin(POSITIVE),
)
print(filtered.num_rows)

KeyboardInterrupt: 